# Ranobe Downloader

Выполняйте шаги по порядку кнопкой ▶. Вставьте ссылку на книгу, выберите перевод, главы и формат.


In [ ]:
# @title 1. Установка
installation = "GitHub" # @param ["GitHub", "Загрузить wheel"]
import importlib.metadata
import subprocess
import sys
from pathlib import Path

PACKAGE_REF = "365ee2839b28b9cf0c7aa44f1d349db5d22b84d0"
if "ranobe_downloader" in sys.modules:
    raise RuntimeError("Пакет уже импортирован. Для переустановки перезапустите среду Colab.")
if installation == "GitHub":
    package = f"ranobe-downloader[images] @ git+https://github.com/SayatYuss/ranobe-downloader.git@{PACKAGE_REF}"
elif installation == "Загрузить wheel":
    from google.colab import files
    uploaded = files.upload()
    wheels = [Path(name) for name in uploaded if Path(name).suffix == ".whl"]
    if len(wheels) != 1:
        raise ValueError("Загрузите один .whl из Releases проекта.")
    package = str(wheels[0].resolve()) + "[images]"
else:
    raise ValueError("Неизвестный способ установки")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
print("Установлено:", importlib.metadata.version("ranobe-downloader"))


In [ ]:
# @title 2. Найти книгу
reference = "" # @param {type:"string"}
authorized = False # @param {type:"boolean"}
from getpass import getpass
from ranobe_downloader import Downloader

book = None
chapters = ()
branch_id = None
saved_path = None
if not reference.strip():
    raise ValueError("Вставьте ссылку на произведение RanobeLib.")
token = getpass("Токен RanobeLib (скрытый ввод): ").strip() if authorized else None
if token and token.lower().startswith("bearer "):
    token = token[7:].strip()
try:
    with Downloader(token=token or None) as downloader:
        book = downloader.get_book(reference.strip())
except Exception:
    raise RuntimeError("Не удалось получить книгу. Проверьте ссылку, доступ к сайту и токен.") from None
print(book.title)
print("Автор:", ", ".join(book.authors) or "не указан")
if book.translations:
    print("Номера переводов для следующего шага:")
    for number, translation in enumerate(book.translations, 1):
        print(f"{number}. {translation.name or 'Перевод'} — {translation.chapter_count} глав (ID {translation.branch_id})")
else:
    print("Отдельных переводов нет. На следующем шаге оставьте номер 1.")


In [ ]:
# @title 3. Выбрать перевод и посмотреть главы
translation_number = 1 # @param {type:"integer"}
show_all_chapters = False # @param {type:"boolean"}
chapters = ()
saved_path = None
if book is None:
    raise ValueError("Сначала выполните шаг 2.")
if type(translation_number) is not int or translation_number < 1:
    raise ValueError("Номер перевода должен быть целым числом от 1.")
if book.translations:
    if translation_number > len(book.translations):
        raise ValueError("Такого номера перевода нет.")
    branch_id = book.translations[translation_number - 1].branch_id
else:
    if translation_number != 1:
        raise ValueError("Для книги без отдельных переводов оставьте номер 1.")
    branch_id = None
chapters = book.list_chapters(branch_id)
if not chapters:
    raise ValueError("В этом переводе нет глав.")
print("Глав в выбранном переводе:", len(chapters))
print("Слева — ПОЗИЦИЯ для выбора диапазона, затем том и номер главы.")
for position, chapter in enumerate(chapters, 1):
    if show_all_chapters or position <= 20 or position > len(chapters) - 5:
        print(f"{position}. Том {chapter['volume']}, глава {chapter['number']} — {chapter['name'] or ''}")
    elif position == 21:
        print("… Для полного списка включите show_all_chapters и запустите шаг снова.")


In [ ]:
# @title 4. Скачать и собрать книгу
scope = "Первые 10" # @param ["Все главы", "Первые 10", "Последние 10", "Диапазон"]
start_position = 1 # @param {type:"integer"}
end_position = 0 # @param {type:"integer"}
output_format = "epub" # @param ["epub", "txt", "html"]
# Позиции применяются только для «Диапазон». Конец 0 означает последнюю главу.
from datetime import datetime, timezone
from tqdm.auto import tqdm

def select_chapter_bounds(chapters, scope, start_position=1, end_position=0):
    if not chapters:
        raise ValueError("Сначала выберите перевод на шаге 3.")
    count = len(chapters)
    if scope == "Все главы":
        start, end = 1, count
    elif scope == "Первые 10":
        start, end = 1, min(10, count)
    elif scope == "Последние 10":
        start, end = max(1, count - 9), count
    elif scope == "Диапазон":
        if type(start_position) is not int or type(end_position) is not int:
            raise ValueError("Позиции должны быть целыми числами.")
        start, end = start_position, end_position or count
        if not 1 <= start <= end <= count:
            raise ValueError(f"Нужен диапазон от 1 до {count}, начало не позже конца.")
    else:
        raise ValueError("Неизвестный режим глав.")
    return chapters[start - 1]["id"], chapters[end - 1]["id"], end - start + 1

saved_path = None
lower, upper, count = select_chapter_bounds(chapters, scope, start_position, end_position)
if output_format not in ("epub", "txt", "html"):
    raise ValueError("Неверный формат.")
output_dir = Path("/content/ranobe-downloads")
output_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
target = output_dir / f"book-{book.id}-{stamp}.{output_format}"
print(book.title, "— выбрано глав:", count)
with tqdm(total=count, unit="глав") as bar:
    def progress(current, total, chapter):
        bar.total = total
        bar.update(max(0, current - bar.n))
    try:
        with Downloader(token=token or None) as downloader:
            result = downloader.download(
                book, branch_id=branch_id, format=output_format, output=target,
                chapter_start_id=lower, chapter_end_id=upper,
                resume=True, cache_dir=output_dir / ".cache", progress=progress,
            )
    except KeyboardInterrupt:
        print("Остановлено. Можно повторить шаг: проверенный кеш сохранён в этой среде.")
    except Exception:
        print("Ошибка загрузки. Проверьте доступ и повторите шаг; подробности можно проверить локально через CLI.")
    else:
        saved_path = Path(result)
        print(f"Готово: {saved_path.name} ({saved_path.stat().st_size / 1024**2:.1f} МиБ)")
print("Недоступные главы могут быть пропущены; завершение не гарантирует полноту книги.")


In [ ]:
# @title 5. Сохранить файл на устройство
from google.colab import files
if saved_path is None or not saved_path.is_file():
    raise ValueError("Нет готового файла. Сначала успешно выполните шаг 4.")
files.download(str(saved_path))


Если файл не скачался, откройте **Files → ranobe-downloads → Download**.

Для другой книги повторите шаги 2–5. Сохраните файл до отключения среды.
